In [ ]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics import confusion_matrix


In [ ]:
data = pd.read_csv("WELFake_Dataset.csv", engine='python', on_bad_lines='warn')

/tmp/ipykernel_302/3378194345.py:1: ParserWarning: Skipping line 329: field larger than field limit (131072)

  data = pd.read_csv("WELFake_Dataset.csv", engine='python', on_bad_lines='warn')
/tmp/ipykernel_302/3378194345.py:1: ParserWarning: Skipping line 6464: field larger than field limit (131072)

  data = pd.read_csv("WELFake_Dataset.csv", engine='python', on_bad_lines='warn')
/tmp/ipykernel_302/3378194345.py:1: ParserWarning: Skipping line 40799: unexpected end of data

  data = pd.read_csv("WELFake_Dataset.csv", engine='python', on_bad_lines='warn')
/tmp/ipykernel_302/3378194345.py:1: ParserWarning: Skipping line 335: Expected 4 fields in line 335, saw 6

  data = pd.read_csv("WELFake_Dataset.csv", engine='python', on_bad_lines='warn')
/tmp/ipykernel_302/3378194345.py:1: ParserWarning: Skipping line 6463: Expected 4 fields in line 6463, saw 7

  data = pd.read_csv("WELFake_Dataset.csv", engine='python', on_bad_lines='warn')
/tmp/ipykernel_302/3378194345.py:1: ParserWarning: Skip

In [ ]:
data.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [ ]:
print(data.shape)
print(data.info())
print(data.describe())
print(data.duplicated().sum()) #zer0 duplicated values in dataset

(40770, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40770 entries, 0 to 40769
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  40770 non-null  object
 1   title       40479 non-null  object
 2   text        40733 non-null  object
 3   label       40750 non-null  object
dtypes: object(4)
memory usage: 1.2+ MB
None
       Unnamed: 0                                              title   text  \
count       40770                                              40479  40733   
unique      40770                                              37395  37426   
top         40749  Factbox: Trump fills top jobs for his administ...          
freq            1                                                  7    392   

        label  
count   40750  
unique      4  
top         1  
freq    21033  
0


In [ ]:
data.isnull().sum()
print(data["label"].value_counts())

label
1                                                  21033
0                                                  19715
 is just unbelievably rich from North Sea oil.         1
 за участие. Благодарю вас.                            1
Name: count, dtype: int64


In [ ]:
# Keep rows with a valid target label
# Do not fill missing labels, because that can teach the model incorrect classes.
data = data.dropna(subset=["label"]).copy()

# Fill missing text fields safely
data["title"] = data["title"].fillna("").astype(str)
data["text"] = data["text"].fillna("").astype(str)


In [ ]:
data.isnull().sum()

,0
Unnamed: 0,0
title,0
text,0
label,0


In [ ]:
data = data.drop(columns=["Unnamed: 0"], errors='ignore')
data["content"] = data["title"] + " " + data["text"]

# Make sure labels are numeric and clean
# WELFake uses: 0 = Fake, 1 = Real
data["label"] = pd.to_numeric(data["label"], errors="coerce")
data = data.dropna(subset=["label"]).copy()
data["label"] = data["label"].astype(int)

data = data[data["label"].isin([0, 1])].copy()

x = data["content"]
y = data["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
#text preprocessing
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
x_train_clean = x_train.apply(clean_text)
x_test_clean = x_test.apply(clean_text)


In [ ]:
vect = CountVectorizer()
x_train_vect = vect.fit_transform(x_train_clean)
x_test_vect = vect.transform(x_test_clean)

In [ ]:
model_builders = {
    "logistic_regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "linear_svc": LinearSVC(class_weight="balanced", random_state=42),
    "passive_aggressive": PassiveAggressiveClassifier(max_iter=1000, random_state=42)
}

trained_models = {}
results = []

for model_name, classifier in model_builders.items():
    candidate = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("clf", classifier)
    ])

    candidate.fit(x_train_clean, y_train)
    candidate_pred = candidate.predict(x_test_clean)

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, candidate_pred),
        "fake_f1": f1_score(y_test, candidate_pred, pos_label=0)
    })
    trained_models[model_name] = candidate

results_df = pd.DataFrame(results).sort_values(["fake_f1", "accuracy"], ascending=False)
display(results_df)

best_model_name = results_df.iloc[0]["model"]
pipeline = trained_models[best_model_name]
y_pred = pipeline.predict(x_test_clean)

print(f"Using best model: {best_model_name}")


,model,accuracy,fake_f1
2,passive_aggressive,0.971288,0.970342
1,linear_svc,0.969325,0.968298
0,logistic_regression,0.952025,0.950662


Using best model: passive_aggressive


In [ ]:
print("accuracy score:",accuracy_score(y_test,y_pred))
print("classification_report:",classification_report(y_test,y_pred))
print(confusion_matrix(y_test, y_pred))

accuracy score: 0.9712883435582822
classification_report:               precision    recall  f1-score   support

           0       0.97      0.97      0.97      3943
           1       0.97      0.97      0.97      4207

    accuracy                           0.97      8150
   macro avg       0.97      0.97      0.97      8150
weighted avg       0.97      0.97      0.97      8150

[[3828  115]
 [ 119 4088]]


In [ ]:
content = input("Enter the news: ")
content_clean = clean_text(content)
prediction = int(pipeline.predict([content_clean])[0])

fake_patterns = [
    r"\bshocking\b",
    r"\bsecret\b",
    r"\bsecretly\b",
    r"\bhidden agenda\b",
    r"\bmind control\b",
    r"\bexperts warn\b",
    r"!{2,}",
    r"\bcures?\b",
    r"\bcompletely cures?\b",
    r"\bno medicine needed\b",
    r"\bmiracle cure\b",
    r"\bscientists confirm\b",
    r"\bfor \d+ days\b",
    r"\b100%\b",
    r"\bguaranteed\b"
]

fake_hits = sum(bool(re.search(pattern, content.lower())) for pattern in fake_patterns)

if fake_hits >= 2:
    prediction = 0
    print("Safety rule applied: fake-news pattern detected.")

label_map = {
    0: "Fake",
    1: "Real"
}

print(label_map[prediction])


Enter the news: Hidden agenda exposed: all ATM machines will stop working forever next week as part of a secret global reset!!!
Safety rule applied: fake-news pattern detected.
Fake


In [ ]:
fak